# Structured Streaming: File Source and Ingestion Directory Monitoring

**Module**: 09_streaming  
**Notebook**: 02_structured_streaming.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Setting up file-based streaming monitoring `/opt/spark-data/streaming/events/`, reading JSON streams, and writing to file sinks.
- **Why does this topic exist?**: Directory monitoring is the standard pattern for landing zones and file-based ETL ingestion.
- **What problem does it solve?**: Processing newly arriving files without re-reading processed files.
- **Where is it used in real-world Data Engineering?**: Automated Bronze layer ingestion in cloud data lakes.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Essential |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | Streaming fundamentals, DataFrame API |
| **Required depth** | Working knowledge: can build a file-source stream and explain its limitations |
| **Data Engineer** | Implement a file-source stream with schema, checkpointing, and monitoring. |
| **Senior Data Engineer** | Diagnose missed files, duplicate processing, and schema drift. |
| **Spark specialist** | Reason about file-source atomicity and directory monitoring. |
| **Next topic** | `09_streaming/03_triggers_and_output_modes.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

The file source watches a directory and processes new files as they appear. It is a convenient way to learn
streaming and to ingest landed files, but it depends on atomic file placement: a reader can see a partially
written file if the producer does not write to a temporary location and rename. The source also does not handle
schema changes automatically.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **File source** | Watch a directory | Processes new files based on modification time and checkpoint |
| **Atomic placement** | Write-then-rename | Prevents readers from seeing partial files |
| **Schema** | Declared stream schema | Must be provided for reliable parsing |
| **Checkpoint** | File progress | Tracks which files have been processed |
| **maxFilesPerTrigger** | Batch size limit | Controls files processed per micro-batch |
| **Late file** | File with an older timestamp | May be ignored depending on the source's tracking |

### How it works, step by step

1. The source lists the directory and filters files by modification time.
2. It compares the list with the checkpoint's processed-file state.
3. New files become the input for the next micro-batch.
4. The files are read with the declared schema.
5. The batch is processed and written to the sink.
6. Processed-file information is committed to the checkpoint.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Directory File Stream Architecture: Watches a directory and atomically processes new files as they arrive.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import os
import time

spark = (
    SparkSession.builder
    .appName("File_Streaming_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
import json

stream_dir = "/opt/spark-data/streaming/events"
checkpoint_dir = "/opt/spark-data/warehouse/stream_chk_02"
out_stream = "/opt/spark-data/warehouse/stream_out_02"

# Clean test dirs
os.system(f"rm -rf {stream_dir}/* {checkpoint_dir} {out_stream}")
os.makedirs(stream_dir, exist_ok=True)

# Define schema for incoming JSON events
stream_schema = StructType([
    StructField("order_id", IntegerType()),
    StructField("customer_id", IntegerType()),
    StructField("amount", DoubleType()),
    StructField("channel", StringType())
])

# Create file stream reader
file_stream = (
    spark.readStream
    .schema(stream_schema)
    .json(stream_dir)
)

print("Streaming DataFrame created from file directory:", file_stream.isStreaming)

---

## Experiment: Generating Files and Observing Stream Pickup

- **1. What are we testing?**: Write sample JSON files into stream_dir and verify the stream picks them up.
- **2. What do we expect?**: Streaming query processes new files incrementally.

In [ ]:
# Start streaming query writing to parquet sink
q = (
    file_stream
    .writeStream
    .format("parquet")
    .option("checkpointLocation", checkpoint_dir)
    .outputMode("append")
    .start(out_stream)
)

# Generate file 1
with open(f"{stream_dir}/event_1.json", "w") as f:
    f.write(json.dumps({"order_id": 1, "customer_id": 101, "amount": 99.5, "channel": "Web"}) + "\n")

time.sleep(3)
q.processAllAvailable()

print(f"Rows processed so far in out_stream: {spark.read.parquet(out_stream).count()}")
q.stop()

**3. What happened?**: Spark detected the new file and wrote output to Parquet.  
**4. Why did it happen?**: Structured Streaming tracks processed file names in the checkpoint directory.  
**5. Practical Engineering Lesson**: File streaming provides checkpointed directory ingestion without an external orchestrator; end-to-end exactly-once still depends on the sink and on atomic file placement.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Landing-zone ingestion

- **Industry**: Logistics
- **Scenario**: Partners drop CSV files into an object-store landing zone throughout the day.
- **Problem**: A batch job runs hourly, so data freshness is poor and late files are missed.
- **Why this concept is relevant**: A file-source stream processes files as they land, with a checkpoint tracking progress.
- **Implementation idea**: Require partners to write to a temp prefix and rename atomically; stream the landing zone into a curated table.
- **Expected result**: Near-continuous ingestion with recoverable progress.
- **Alternative**: Use a queue or event notification instead of directory listing.
- **Trade-offs**: A queue is more scalable and avoids listing costs but requires producer integration.

### Use Case 2: Schema drift in partner files

- **Industry**: Retail
- **Scenario**: A partner adds a column to a daily CSV without notice.
- **Problem**: The stream's declared schema ignores the new column, and a changed type can break parsing.
- **Why this concept is relevant**: A schema contract and validation step detect drift before it reaches the curated table.
- **Implementation idea**: Parse with an explicit schema, validate headers/types, and quarantine non-conforming files.
- **Expected result**: A pipeline that fails visibly instead of silently dropping data.
- **Alternative**: Schema inference per file.
- **Trade-offs**: Inference hides drift but makes the stream schema unstable and expensive.

---

## Business Value

- Fresher data for operational teams.
- Lower latency than hourly batch jobs.
- Recoverable file processing.
- Explicit contracts with data providers.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Require atomic file placement.
- Declare the schema explicitly.
- Monitor file arrival latency and batch duration.
- Bound batch size with `maxFilesPerTrigger`.
- Plan for schema drift and quarantine.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Stream a landing directory

**Steps**

1. Create a stream over a directory with a declared schema.
2. Add a complete file and observe the batch.
3. Add a partially written file and observe the failure or corrupt read.

**What to observe**: Atomic placement is required; a partial file can be read before it is complete.

**Healthy result**: Only complete files are processed.

**Problematic result**: The stream picks up a temp file and produces corrupt rows.

**Engineering takeaway**: The producer's write protocol is part of the streaming contract.

---

## Trade-offs and Alternatives

### Advantages

- Simple setup
- Works with existing landing zones
- Checkpointed progress

### Limitations

- Directory listing cost
- Partial-file risk
- Limited schema evolution

### When to use it

- Partner file ingestion
- Log landing zones
- Learning streaming

### When not to use it

- High-file-count directories
- Non-atomic producers
- Unbounded schema drift

### Alternatives inside Spark

- File source
- Checkpoint
- Schema enforcement

### Alternatives outside Spark

- Kafka
- Object-store notifications
- Managed ingestion services

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Inspect Checkpoint Directory
**Problem Statement**: List contents of checkpoint directory.  
**Input / Context**: checkpoint_dir

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Add Second Event File
**Problem Statement**: Write a second file and verify incremental addition.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Atomic File Renaming
**Problem Statement**: Why files should be moved into directory atomically.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The stream that missed yesterday's file

**Scenario**: A file arrives with an older modification time and the stream never processes it.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check the file timestamp and the source's tracking behavior.
2. Verify the checkpoint's processed-file state.
3. Backfill the file explicitly or adjust the producer's naming/timestamp convention.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Inspect Checkpoint Directory):
```python
print(os.listdir(checkpoint_dir))
```

#### Solution 2 (Add Second Event File):
```python
print('File streaming is incremental.')
```

#### Solution 3 (Atomic File Renaming):
```python
print('Writing directly to directory risks Spark reading partial half-written files.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **File Stream**: Monitors directory for new files.
- **Checkpoint**: Records processed file offsets.

### Essential Technical Terms
- `processAllAvailable()`: Processes pending data synchronously (great for testing).
- `checkpointLocation`: Storage for progress metadata.

### Common Engineering Mistakes
- **Writing files directly into stream folder**: Causes partial read errors; always write to temp and atomic mv.

### Interview Questions on This Topic
- **Q: How does Structured Streaming avoid reprocessing files?**
  - *A*: It tracks processed file paths and versions in the internal offset log inside the checkpoint directory.


### More Interview Questions

- **Q: Why does the file source require atomic file placement?**
  - *A*: The source discovers files by listing; if a producer writes directly to the final path, the reader can see a partial file before the write completes.
- **Q: How does the file source track progress?**
  - *A*: It records processed files in the checkpoint and compares the directory listing against that state on each batch.

### What to Study Next
Open 03_triggers_and_output_modes.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')